In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# 1. Load and clean data
data = pd.read_csv(
    'website_traffic.csv',
    parse_dates=['Date'],
    index_col='Date'
)

data = data.sort_index()
data = data.interpolate(method='time')

print("First five rows:")
print(data.head())

print("\nMissing values:")
print(data.isnull().sum())

# 2. Plot website traffic
data[['Page_Views', 'Unique_Visitors', 'Bounce_Rate']].plot(
    subplots=True, figsize=(10, 8), title="Website Traffic Analysis"
)
plt.tight_layout()
plt.show()

# 3. Time series decomposition
decomposition = seasonal_decompose(
    data['Page_Views'],
    model='additive',
    period=7
)

decomposition.plot()
plt.suptitle("Time Series Decomposition", y=1.02)
plt.tight_layout()
plt.show()

# 4. Moving average
data['Page_Views_MA'] = data['Page_Views'].rolling(
    window=30, min_periods=1
).mean()

plt.figure(figsize=(10, 5))
plt.plot(data.index, data['Page_Views'], label='Page Views')
plt.plot(data.index, data['Page_Views_MA'],
         label='30-Day Moving Average', color='red')
plt.title("Page Views with Moving Average")
plt.xlabel("Date")
plt.ylabel("Page Views")
plt.legend()
plt.tight_layout()
plt.show()

# 5. Seasonal plot: Average traffic by day of week
data['Day'] = data.index.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday',
             'Thursday', 'Friday', 'Saturday', 'Sunday']

seasonal_data = data.groupby('Day')['Page_Views'].mean()
seasonal_data = seasonal_data.reindex(day_order)

seasonal_data.plot(kind='bar', figsize=(9, 5))
plt.title("Average Page Views by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Page Views")
plt.tight_layout()
plt.show()

# 6. Anomaly detection using Z-score
mean = data['Page_Views'].mean()
std = data['Page_Views'].std()

data['Page_Views_Z'] = (
    data['Page_Views'] - mean
) / std

anomalies = data[data['Page_Views_Z'].abs() > 3]

print("\nDetected anomalies:")
print(anomalies[['Page_Views', 'Page_Views_Z']])

# 7. Visualize anomalies
plt.figure(figsize=(10, 5))
plt.plot(data.index, data['Page_Views'],
         label='Page Views')
plt.scatter(anomalies.index, anomalies['Page_Views'],
            color='red', label='Anomalies')

plt.title("Website Traffic Anomaly Detection")
plt.xlabel("Date")
plt.ylabel("Page Views")
plt.legend()
plt.tight_layout()
plt.show()